# 0. Context
Feature importance is the *feature* that gets me hooked on tree-based methods.

Two types of feature importance can be extracted from LightGBM boosters: *split* and *gain*. Referring to 
https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.Booster.html#lightgbm.Booster.feature_importance:
> importance_type (string, optional (default="split")) – How the importance is calculated. If “split”, resultcontains numbers of times the feature is used in a model. If “gain”, result contains total gains of splits which use the feature.

There is often a confusion between the two importance types: *split* vs *gain*. Here is a look-inside, back-of-envelop demo of the two:
1. start with output from *booster.dump_model()['tree_info']*;
2. calculate feature importance of type *split*;
3. prove perfect tally with output from *booster.feature_importance()*;
4. repeat the same steps for feature importance of type *gain*.

# 1. Initiation rite
Invocations we can't go on without.

In [1]:
import pandas as pd
import lightgbm as lgb

# 2. Toy data & toy model
Borrowing data from https://www.kaggle.com/uciml/iris.
*Replace next cell with your own data and model.*

In [2]:
import os
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split

train = pd.read_csv('/kaggle/input/iris/Iris.csv', index_col='Id')
dataX = train[ train.columns[train.columns.str.contains('Cm$')] ]
datay = OrdinalEncoder().fit_transform(train[['Species']]).flatten().astype(int)
trainSet = lgb.Dataset(dataX, datay)

param = {'objective'       : 'multiclass',
         'metric'          : 'multi_logloss',
         'num_class'       : train['Species'].nunique(),
         'num_leaves'      : 3,
         'min_data_in_leaf': 36, 
         'learning_rate'   : .15,
         'num_boost_round' : 30}
model = lgb.train(param, trainSet)

/opt/conda/lib/python3.7/site-packages/lightgbm/engine.py:148: UserWarning: Found `num_boost_round` in params. Will use it instead of argument
  warnings.warn("Found `{}` in params. Will use it instead of argument".format(alias))


# 3. Put trees in Pandas DataFrame
Skip the next cell unless you are particularly interested! The original notebook where I wrote these[](http://) functions is available from https://www.kaggle.com/marychin/lightgbm-trees-to-pandas-dataframe.

In [3]:
def grabdict(tisdict, tree_index, split_index, depth, splits, leaves):
# recursive function to unravel nested dictionaries
    depth += 1
    if 'split_index' in tisdict.keys():
        tis = tisdict.copy()
        del tis['left_child']
        del tis['right_child']
        tis['tree_index'] = tree_index
        split_index = tis['split_index']
        splits = pd.concat([splits, pd.DataFrame(tis, index=[len(splits)])])
        splits, leaves = grabdict(tisdict['left_child'], tree_index, split_index, depth, splits, leaves)
        splits, leaves = grabdict(tisdict['right_child'], tree_index, split_index, depth, splits, leaves)
    else:
        tis = tisdict.copy()
        tis['tree_index'] = tree_index
        tis['split_index'] = split_index
        tis['depth'] = depth
        leaves = pd.concat([leaves, pd.DataFrame(tis, index=[len(leaves)])])
    return splits, leaves

def grabtrees(model):
# wrapper function to call the two functions above
    splits, leaves = pd.DataFrame(), pd.DataFrame()
    tree_info = model.dump_model()['tree_info']
    for tisdict in tree_info:
        splits, leaves = grabdict(tisdict['tree_structure'], tisdict['tree_index'], 0, 0, splits, leaves)
    leaves = leaves.merge(splits, left_on=['tree_index', 'split_index'], right_on=['tree_index', 'split_index'], how='left')
    return tree_info, leaves

tree_info, leaves = grabtrees(model)
leaves   # all leaves in a single df: one leaf per row

,leaf_index,leaf_value,leaf_weight,leaf_count,tree_index,split_index,depth,split_feature,split_gain,threshold,decision_type,default_left,missing_type,internal_value,internal_weight,internal_count
0,0,-0.880230,22.666667,51,0,0,2,2,72.794098,3.15,<=,True,None,0.000000,0.00000,150
1,1,-1.211112,44.000000,99,0,0,2,2,72.794098,3.15,<=,True,None,0.000000,0.00000,150
2,0,-1.211112,21.333333,48,1,0,2,2,17.647100,1.80,<=,True,None,0.000000,0.00000,150
3,1,-0.911112,24.000000,54,1,1,3,3,41.040401,1.65,<=,True,None,0.352941,45.33330,102
4,2,-1.197050,21.333333,48,1,1,3,3,41.040401,1.65,<=,True,None,0.352941,45.33330,102
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
264,2,0.035329,7.042271,67,88,1,3,0,0.437202,5.15,<=,True,None,0.116708,9.11514,108
265,1,-0.038361,2.884964,42,88,0,2,2,0.303985,5.05,<=,True,None,0.000000,0.00000,150
266,0,-0.076539,1.213377,36,89,1,3,1,0.000018,3.05,<=,True,None,-0.508155,1.73031,78
267,2,-0.075481,0.516935,42,89,1,3,1,0.000018,3.05,<=,True,None,-0.508155,1.73031,78


# 4. importance_type='split'

In [4]:
imp_split = model.feature_importance('split')
for tt in range(len(imp_split)):
    print(imp_split[tt], model.feature_name()[tt])
# So PetalLengthCm is found to be most important (by 'split' sense)
# followed by PetalWidthCm, SepalWidthCm and lastly SepalLengthCm.

15 SepalLengthCm
22 SepalWidthCm
81 PetalLengthCm
61 PetalWidthCm


### Where do these values come from?

In [5]:
# Recall that DataFrame *leaves* contain one leaf per row.
# Each split produces 2 leaves sharing a common splitting criterion.
# We should therefore first drop the duplicating partner-leaves.
one_split_per_row = leaves[['tree_index', 'split_index', 'split_feature', 'split_gain']].drop_duplicates()
one_split_per_row

,tree_index,split_index,split_feature,split_gain
0,0,0,2,72.794098
2,1,0,2,17.647100
3,1,1,3,41.040401
5,2,1,2,0.647059
7,2,0,3,62.040401
...,...,...,...,...
261,87,1,1,0.000572
263,88,1,0,0.437202
265,88,0,2,0.303985
266,89,1,1,0.000018


In [6]:
one_split_per_row['split_feature'].value_counts(sort=False)
# voila; we reproduce the exact same numbers of model.feature_importance('split')

0    15
1    22
2    81
3    61
Name: split_feature, dtype: int64

# 5. importance_type='gain'

In [7]:
imp_gain = model.feature_importance('gain')
for tt in range(len(imp_gain)):
    print(imp_gain[tt], model.feature_name()[tt])
# So PetalLengthCm is found to be most important (by 'gain' sense)
# followed by PetalWidthCm, SepalWidthCm and lastly SepalLengthCm.
# In this problem the order of feature importance happens to agree in both 'split' and 'gain' senses.
# This is not always the case.

2.1440600478308625 SepalLengthCm
4.087134801840875 SepalWidthCm
707.3842577012983 PetalLengthCm
360.7060055512011 PetalWidthCm


### Where do these floating-point values come from?

In [8]:
one_split_per_row.groupby('split_feature')['split_gain'].sum()
# Voila: perfect tally!

split_feature
0      2.144060
1      4.087135
2    707.384258
3    360.706006
Name: split_gain, dtype: float64

# 6. Sister notebooks: the Leaf-by-leaf series
Decision trees: a leaf-by-leaf demo

https://www.kaggle.com/marychin/decision-trees-a-leaf-by-leaf-demo

**num_leaves** and **min_data_in_leaf**: a LightGBM demo

https://www.kaggle.com/marychin/num-leaves-min-data-in-leaf-a-lightgbm-demo

min_sum_hessian: a LightGBM demo

https://www.kaggle.com/marychin/min-sum-hessian-a-lightgbm-demo

feature_importances split vs gain: a demo (we are here)

https://www.kaggle.com/marychin/feature-importances-split-vs-gain-a-demo

# 7. Cheers, Kagglers & Kaggle!
Together we democratise learning and skills.